In [1]:
from google.colab import files
uploaded = files.upload()

Saving ev_charging_dataset.xlsx to ev_charging_dataset.xlsx


In [2]:
import pandas as pd

all_sheets = pd.read_excel('ev_charging_dataset.xlsx', sheet_name=None)
sessions_df = all_sheets['sessions']

print(sessions_df.shape)

(294024, 15)


In [3]:
customer_features = sessions_df.groupby('customer_id').agg(
    frequency=('session_id', 'count'),
    avg_spend=('total_cost', 'mean'),
    avg_energy=('energy_kwh', 'mean')
).reset_index()

print(customer_features.head())
print(customer_features.shape)

  customer_id  frequency  avg_spend  avg_energy
0  CUST_00001        267  27.182996   46.066667
1  CUST_00002        243  27.917778   47.815638
2  CUST_00003        282  27.729397   46.678014
3  CUST_00004        253  28.151383   48.001581
4  CUST_00005        246  27.976626   47.614228
(8000, 4)


In [4]:
from sklearn.preprocessing import StandardScaler

features_to_use = customer_features[['frequency', 'avg_spend', 'avg_energy']]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_use)

print(scaled_features[:5])

[[ 4.06118986e+00  4.02708826e-03 -1.63678771e-01]
 [ 3.63786814e+00  1.53500232e-01  2.78275374e-02]
 [ 4.32576594e+00  1.15178851e-01 -9.67383281e-02]
 [ 3.81425219e+00  2.01021523e-01  4.81876723e-02]
 [ 3.69078335e+00  1.65471452e-01  5.77381377e-03]]


In [5]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
customer_features['cluster'] = kmeans.fit_predict(scaled_features)

print(customer_features.head())
print(customer_features['cluster'].value_counts())

  customer_id  frequency  avg_spend  avg_energy  cluster
0  CUST_00001        267  27.182996   46.066667        2
1  CUST_00002        243  27.917778   47.815638        2
2  CUST_00003        282  27.729397   46.678014        2
3  CUST_00004        253  28.151383   48.001581        2
4  CUST_00005        246  27.976626   47.614228        2
cluster
0    6237
1    1363
2     400
Name: count, dtype: int64


In [6]:
cluster_summary = customer_features.groupby('cluster').agg(
    avg_frequency=('frequency', 'mean'),
    avg_spend=('avg_spend', 'mean'),
    avg_energy=('avg_energy', 'mean'),
    customer_count=('customer_id', 'count')
).reset_index()

print(cluster_summary)

   cluster  avg_frequency  avg_spend  avg_energy  customer_count
0        0      29.376944  28.769388   49.950888            6237
1        1       5.721937  19.621648   36.789092            1363
2        2     257.502500  27.816552   47.011896             400


In [7]:
cluster_summary['total_segment_revenue'] = cluster_summary['avg_spend'] * cluster_summary['avg_frequency'] * cluster_summary['customer_count']

print(cluster_summary[['cluster', 'customer_count', 'total_segment_revenue']])

   cluster  customer_count  total_segment_revenue
0        0            6237           5.271242e+06
1        1            1363           1.530292e+05
2        2             400           2.865133e+06


In [12]:
customer_features.to_csv('customer_segments.csv', index=False)
cluster_summary.to_csv('cluster_summary.csv', index=False)

from google.colab import files
files.download('customer_segments.csv')
files.download('cluster_summary.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>